# Lesson 13 — Coherent is not correct, and only settled markets can tell you

A price vector can be perfectly coherent and perfectly wrong, because a Dutch-book test never compares a price to the world. Scoring settled markets splits the error into a part a recalibration can repair and a part that is a property of the question.

**The rule.** `Brier = Reliability − Resolution + Uncertainty + Binning`

**When it holds.** Over a corpus of forecasts quoted well before close and scored against what actually happened.

**When it fails.** Scoring last traded prices. A last trade happens moments before settlement when the answer is largely known, so it scores near-perfectly and measures how fast the exchange converges rather than whether it saw anything coming. The corpus is also whatever the venue lists most, which is not a sample.

| | |
|---|---|
| Lesson id | `calibration` |
| Pane it appears on | `calibration` (panes carry more than one lesson) |
| Code it is about | `modules/coherence/kernel/calibration.py`, `modules/coherence/fs/corpus.py` |
| Tests that go red if it stops being true | `tests/test_coherence_calibration.py` |
| Pane shipped | yes |

Every cell below runs against the real kernel. Nothing here is a re-implementation:
a number this notebook prints is the number the engine would produce for the same
input. The recorded Kalshi payloads come from `tests/fixtures/coherence/`.

In [ ]:
import json
import sys
from decimal import Decimal
from pathlib import Path

# This notebook lives in notebooks/coherence_lab/ and imports the kernel two
# levels up. Found by walking upward rather than by counting parents, so the
# notebook runs from its own directory or from Part2_Infrastructure.
HERE = Path.cwd().resolve()
ROOT = next((path for path in (HERE, *HERE.parents) if (path / "modules" / "coherence" / "kernel").is_dir()), None)
if ROOT is None:
    raise SystemExit(f"no coherence kernel above {HERE}: open this notebook from inside Part2_Infrastructure")
sys.path.insert(0, str(ROOT))

FIXTURES = ROOT / "tests" / "fixtures" / "coherence"


def fixture(name: str) -> dict:
    """One recorded Kalshi response, envelope and all, exactly as it was sent.

    These are captures, not mocks. Where a number below looks odd it is because
    the exchange quoted it, and `tools/capture_kalshi_fixtures.py` re-records
    them.
    """
    return json.loads((FIXTURES / f"{name}.json").read_text(encoding="utf-8"))


print(f"kernel root       {ROOT}")
print(f"recorded fixtures {FIXTURES.is_dir()}")

## 1. A settled corpus with the favourite-longshot shape in it

In [ ]:
from modules.coherence.fs.corpus import Settlement, final_trade_forecasts
from modules.coherence.kernel import calibration

# Ten price bands, two hundred settled markets each, with the oldest empirical
# finding in this literature built in: longshots happen less often than they are
# priced and favourites more often.
BANDS = (
    ("0.05", 200, 4), ("0.15", 200, 22), ("0.25", 200, 44), ("0.35", 200, 64), ("0.45", 200, 86),
    ("0.55", 200, 116), ("0.65", 200, 138), ("0.75", 200, 160), ("0.85", 200, 180), ("0.95", 200, 196),
)
corpus = [
    calibration.Forecast(f"KXDEMO-{price}-{index}", "KXDEMO", Decimal(price), index < hits, 3_600)
    for price, count, hits in BANDS
    for index in range(count)
]
report = calibration.score(corpus, engine="tape")

print(f"  {report.count} settled markets, quoted an hour before close, base rate {report.base_rate}")
print()
CENTS = Decimal("0.0001")
print("  price band     n     priced at   happened   deviation")
for band in report.bins:
    if not band.count:
        continue
    print(
        f"  {band.label:<13} {band.count}    {band.mean_forecast.quantize(CENTS)}      "
        f"{band.outcome_rate.quantize(CENTS)}     {band.deviation.quantize(CENTS)}"
    )
print()
print("  Negative at the bottom, positive at the top: longshots overpriced, favourites")
print("  underpriced. That is the favourite-longshot shape, and it is the reason the")
print("  slope below comes out above one rather than below it.")

## 2. Murphy's decomposition, exactly

In [ ]:
rebuilt = report.reliability - report.resolution + report.uncertainty + report.binning
PLACES = Decimal("0.0000001")
print(f"  Brier         {report.brier.quantize(PLACES):f}")
print(f"  Reliability   {report.reliability.quantize(PLACES):f}")
print(f"  Resolution    {report.resolution.quantize(PLACES):f}")
print(f"  Uncertainty   {report.uncertainty.quantize(PLACES):f}")
print(f"  Binning       {report.binning.quantize(PLACES):f}   (zero here: every band holds one price, so nothing was discarded)")
print()
print("  and unquantised, which is what the identity is checked on:")
print(f"  reliability - resolution + uncertainty + binning = {rebuilt}")
print(f"  Brier                                           = {report.brier}")
print(f"  EXACTLY equal, as Decimals: {rebuilt == report.brier}")
print("  (Decimal equality compares value, not exponent, so 0.1492000 and 0.1492 are one")
print("  number written two ways — the trailing zeros are the arithmetic's, not a rounding.)")
print()
print("  Not equal to six decimal places — equal. Every count here divides exactly, so")
print("  nothing rounds, and the identity is arithmetic rather than a numerical accident.")
print()
print("  Reliability is the only term a recalibration repairs. Resolution enters with a")
print("  minus sign, so it is the term you want large: a forecaster who quotes the base")
print("  rate on every market is perfectly reliable and useless, and only resolution")
print("  notices. Uncertainty is a property of the question, which is why raw Brier scores")
print("  are not comparable across corpora and are never reported here without the split.")
print()
print(f"  skill against always quoting the base rate: {report.skill.quantize(Decimal('0.0001'))}")

## 3. The slope, and which way it should lean

In [ ]:
print(f"  weighted least-squares slope of outcome rate on price: {report.bias_slope.quantize(Decimal('0.000001'))}")
print(f"  above one: {report.bias_slope > Decimal(1)}")
print()
print("  The direction is worth deriving rather than remembering. Longshots are overbet,")
print("  so a 5-cent contract happens less than 5% of the time and its point sits BELOW")
print("  the diagonal. Favourites are underbet, so a 95-cent contract happens more than")
print("  95% of the time and sits ABOVE it. A line through a scatter pulled down at the")
print("  left and up at the right is STEEPER than the diagonal, not shallower.")
print()
print("  reported with the bin counts, because on a thin corpus this is mostly noise:")
print(f"    {report.count} markets across {len(report.composition)} series, thin = {report.thin}")

## 4. The recalibration map has to be non-decreasing

In [ ]:
print("  quoted    calibrated    weight")
previous = None
monotone = True
for point in report.isotonic_map:
    print(f"  {point.quoted:f}    {point.calibrated.quantize(Decimal('0.0001')):f}        {point.weight}")
    if previous is not None and point.calibrated < previous:
        monotone = False
    previous = point.calibrated
print()
print(f"  non-decreasing across all {len(report.isotonic_map)} points: {monotone}")
print()
print("  It has to be. A higher price mapping to a lower probability would make the")
print("  CORRECTED prices themselves incoherent, and this engine would be shipping the")
print("  fault it exists to find. Pool-adjacent-violators is the exact solution, not an")
print("  approximation: whenever a block dips below the one before it, the two merge into")
print("  their weighted mean and the check runs backwards again.")
print()

# Nothing pooled above, because the quoted curve was already monotone. Invert one
# band and the machinery becomes visible.
INVERTED = tuple(
    (price, count, 150 if price == "0.45" else hits) for price, count, hits in BANDS
)
pooled = calibration.score(
    [
        calibration.Forecast(f"KXINV-{price}-{index}", "KXINV", Decimal(price), index < hits, 3_600)
        for price, count, hits in INVERTED
        for index in range(count)
    ],
    engine="tape",
)
print(f"  with the 0.45 band inverted, the map collapses from {len(report.isotonic_map)} points to {len(pooled.isotonic_map)}:")
for point in pooled.isotonic_map:
    print(f"    {point.quoted:f} -> {point.calibrated.quantize(Decimal('0.0001')):f} (weight {point.weight})")

## 5. The trap: scoring the answer instead of the forecast

In [ ]:
# The trap. `GET /markets?status=settled` returns a last traded price for markets
# that have already resolved. It is public, instant, and nearly worthless.
settlements = [
    Settlement(
        ticker=f"KXSETTLED-{index}",
        event_ticker="KXSETTLED",
        series_ticker="KXSETTLED",
        close_ts_ns=None,
        result="yes" if index % 2 == 0 else "no",
        last_price=Decimal("0.9800") if index % 2 == 0 else Decimal("0.0200"),
    )
    for index in range(120)
]
final = calibration.score(final_trade_forecasts(settlements), engine="final_trade")

print(f"  corpus size {final.count}, thin = {final.thin}, median horizon {final.median_horizon_s}s")
print()
print(f"  forecasts quoted an hour out : Brier {report.brier}   skill {report.skill.quantize(Decimal('0.0001'))}")
print(f"  last traded prices           : Brier {final.brier}   skill {final.skill.quantize(Decimal('0.0001'))}")
ratio = (report.brier / final.brier).quantize(Decimal("1"))
print()
print(f"  The second engine scores {ratio} times better and the number means nothing.")
print("  A last trade happens seconds before settlement, when the answer is largely known,")
print("  so it measures how fast the exchange converges rather than whether it saw")
print("  anything coming. It is not a thin-sample problem either — this corpus clears the")
print("  floor, and no amount more of it would help.")
print()
print("  which is why the report says so in its own words rather than scoring itself well:")
for note in final.detail.split("; "):
    print(f"    {note}")